In [ ]:
!pip install rdkit

# From A Supplier Catalogue To A Drug-Likeness Verdict

A chemical supplier has sent a catalogue of nine chemicals we can use in our research, each as a [SMILES](https://en.wikipedia.org/wiki/Simplified_Molecular_Input_Line_Entry_System) string typed by hand.
One string does not parse, four of the records concern only two compounds because the supplier listed single enantiomers and racemates side by side, and half the [stereocentres](https://en.wikipedia.org/wiki/Stereocenter) in the file carry no configuration at all.
Today you repair the broken SMILES string, decide once and for all whether this dataset keeps or discards stereochemistry, turn the surviving structures into a descriptor table, and finish with a drug-likeness verdict on every compound from two different measures.

By the end the notebook prints something like this:

```
                   name     mw  logp  violations  lipinski    qed
0          naproxen (S)  230.3  3.04           0         1  0.881
2            ketoprofen  254.3  3.11           0         1  0.853
1  ibuprofen (racemate)  206.3  3.07           0         1  0.822
5            loratadine  382.9  4.89           0         1  0.705
3            salbutamol  239.3  1.31           0         1  0.639
4               aspirin  180.2  1.31           0         1  0.550
6            amiodarone  645.3  6.94           2         0  0.168
```


## Part 1 - Reading The Catalogue

Everything today runs on [RDKit](https://www.rdkit.org/docs/) plus [pandas](https://pandas.pydata.org/docs/).


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, QED
from rdkit.Chem.Draw import IPythonConsole

The catalogue is below, exactly as it arrived.
Six of the nine are [profen](https://en.wikipedia.org/wiki/Nonsteroidal_anti-inflammatory_drug)-type anti-inflammatories or small analgesics.
The last two, [loratadine](https://en.wikipedia.org/wiki/Loratadine) and [amiodarone](https://en.wikipedia.org/wiki/Amiodarone), are heavier and greasier, and are there to push the drug-likeness filter in Part 4 to its limits.

| catalogue name | what the supplier claims |
|---|---|
| naproxen (S) | the marketed single enantiomer |
| naproxen (R) | the distomer, far weaker as a COX inhibitor |
| ibuprofen (racemate) | sold as the 1:1 mixture |
| ibuprofen (S) | the active enantiomer alone |
| ketoprofen | sold as the racemate, string typed by hand |
| salbutamol | a chiral aminoalcohol, configuration not stated |
| aspirin | achiral |
| loratadine | achiral antihistamine |
| amiodarone | achiral antiarrhythmic, very lipophilic |


The catalog is stored as a dictionary below.

In [ ]:
supplier_catalogue = {
    "naproxen (S)": "C[C@@H](c1ccc2cc(OC)ccc2c1)C(=O)O",
    "naproxen (R)": "C[C@H](c1ccc2cc(OC)ccc2c1)C(=O)O",
    "ibuprofen (racemate)": "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "ibuprofen (S)": "CC(C)Cc1ccc(cc1)[C@H](C)C(=O)O",
    "ketoprofen": "CC(c1cccc(c1)C(=O)c1ccccc1C(=O)O",
    "salbutamol": "CC(C)(C)NCC(O)c1ccc(O)c(CO)c1",
    "aspirin": "CC(=O)Oc1ccccc1C(=O)O",
    "loratadine": "CCOC(=O)N1CCC(=C2c3ccc(Cl)cc3CCc3cccnc32)CC1",
    "amiodarone": "CCCCc1oc2ccccc2c1C(=O)c1cc(I)c(OCCN(CC)CC)c(I)c1",
}
print(len(supplier_catalogue), "records")

#### Questions
1. Read the aspirin SMILES string `CC(=O)Oc1ccccc1C(=O)O` through in your head, atom by atom, the way [video 1](https://youtu.be/56YNm3nZ_9o) read `CC(=O)O` aloud and record the atoms, the functional groups and so on.
2. Name the two atoms that the ring-closure digit `1` joins, and say what hangs inside each pair of parentheses.


### A Cleaning Function
Turning a list of hand-typed SMILES into one clean row per structure is the job we did in track 2: Parse every string, report the ones that fail, and keep only the first record of each structure.

Today that cleaning of data is handed over as a finished function (a little program) called `clean_catalogue` defined below.
Today is not about cleaning the data, but on evaluating chemistry- and pharmaceutical-related problems instead.

---

**NB!** Handed over is not the same as ignored, though: you will be writing code like this again after the fall holiday, and you should be able to say what every line of it does.

Two things in it are worth knowing before you read it.
A [`set`](https://docs.python.org/3/tutorial/datastructures.html#sets) stores each value only once, so asking whether a key is already in `seen` is how the function tells a first occurrence from a repeat.
And `Chem.MolFromSmiles` does not raise an error when it cannot read a string: it returns `None`, which is why the function checks for it before doing anything else with the molecule.

---


In [ ]:
def clean_catalogue(catalogue, keep_stereo):
    """Parse every record in a catalogue

    Uses the same infrastructure that we worked on in Track 2 of the FA516 course.

    Reports failures, and keeps only one row per structure.

    :param catalogue: a dictionary of molecule names and their SMILES string
    :param keep_stereo: True or False - whether to keep the stereochemistry.
    :returns: a Pandas dataframe with the fixed data.
    """
    rows = []
    seen_smiles = set()
    for name, smiles in catalogue.items():
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            print(f"Molecule {name:s} has invalid SMILES: '{smiles:s}'")
        else:
            canonical_smiles = Chem.MolToSmiles(mol, isomericSmiles=keep_stereo)
            if canonical_smiles in seen_smiles:
                print(f"Duplicate {name:s} skipped because it was stored earlier.")
            else:
                seen_smiles.add(canonical_smiles)
                rows.append({"name": name, "smiles": canonical_smiles, "mol": mol})

    print(f"Initial catalog had {len(catalogue):d} records. Cleaned dataset has {len(rows):d} unique structures.")
    return pd.DataFrame(rows)

#### Questions

3. Read the function `clean_catalogue` above and *explain* in your own words, what happens in the code. Compare your answer with [previous weeks](https://colab.research.google.com/drive/1VmDZpV_yLvI3OBqW5Wl0nCojRGCHYYBD#scrollTo=vHaT388mZPeC) work.

  - 3.1 What happens to a record whose SMILES does not parse?
  - 3.2 What decides whether a record counts as a duplicate?
  - 3.3 What does the `keep_stereo` argument changes about that decision.
  
4. Predict how many rows of data setting `keep_stereo=True`, and run the block below to check.
5. How many if you set it to false? Explain the difference by printing the `first_pass` variable.


In [ ]:
initial_dataset = clean_catalogue(_, keep_stereo=_)

[Ketoprofen](https://en.wikipedia.org/wiki/Ketoprofen) is 2-(3-benzoylphenyl)propanoic acid: a propanoic acid whose 2-carbon carries a benzene ring, which in turn carries a benzoyl group at the *meta* position.
Count the opening and closing parentheses in the supplier's string and you will find they do not balance, so the branch holding the aromatic system is never closed.

### Question

6. Write a corrected SMILES for ketoprofen into the catalogue, parse it, and confirm the parse succeeded by drawing it.


In [ ]:
supplier_catalogue["ketoprofen"] = "_"
ketoprofen = Chem.MolFromSmiles(supplier_catalogue["ketoprofen"])
print("ketoprofen parsed:", ketoprofen is not None)
ketoprofen

### Updating the Dataset
We update the supplier catalogue and we verify that everything now converts cleanly by converting the catalogue to our `initial_dataset`.

In [ ]:
initial_dataset = clean_catalogue(_, keep_stereo=True)

#### Questions
7. The `initial_dataset` is a pandas `DataFrame`. Show it.

### Visualize all Molecules
With every string parsing, visualize at all nine structures at once before deciding anything about them using the `Draw.MolsToGridImage` function from RDKit.


In [ ]:
molecules = list(initial_dataset["_"])
names = list(initial_dataset["_"])
Draw.MolsToGridImage(molecules, legends=names, molsPerRow=3, subImgSize=(220, 180))

#### Questions

7. Look at the grid and at the four profen structures in particular. Which pairs of pictures are nearly impossible to tell apart, and what does that say about how many distinct data points the nine records really contain?


## Part 2 - One Decision About Stereochemistry

[Enantiomers](https://en.wikipedia.org/wiki/Enantiomer) have identical atoms and identical bonds, and a chiral binding site still tells them apart.
[S-naproxen inhibits COX](https://doi.org/10.1074/jbc.M110.162982) and the R form is roughly a hundredfold weaker and contributes mostly to the liver's workload.
S-ibuprofen is the active form too, though the body converts much of the R form into it, which is why the racemate is still sold.

So this is a real pharmacological distinction, and yet a dataset can only keep it if the records actually carry the markers.
[Video 2](https://youtu.be/qqr-nnmF_Rc)s rule was: decide once, write the decision down, and apply it to the whole table.
The next few blocks gather the evidence for that decision.


#### Questions

1. Write the loop that walks the catalogue `names` and the `molecules` list together with `zip` and prints, for each compound, its name followed by `Chem.FindMolChiralCenters(mol, includeUnassigned=True)` ([documentation for function](https://www.youtube.com/watch?v=Of8MPuVeiMw)). *Hint:* see [video 1.17](https://www.youtube.com/watch?v=Of8MPuVeiMw) again if you are in doubt.
2. Read the output: which compounds report a centre labelled `?`, and what does that label mean about the string the supplier typed?

In [ ]:
for name, mol in zip(_, _):
    print(name, Chem.FindMolChiralCenters(_, includeUnassigned=True))

Three centres specified and three unspecified is a suspicious ratio for a file that claims to be selling single enantiomers.
The helper's `keep_stereo` argument is passed straight on to `Chem.MolToSmiles` as `isomericSmiles`, and `isomericSmiles=False` writes *only* the connectivity and nothing else.
Before deciding, it is worth being sure what that does to the naproxen pair.

#### Questions

3. Predict what each of the two lines with `print` prints for the naproxen pair, then complete the second line and run it.
4. On the survey above, on this comparison and on video 2's criterion, argue whether this dataset should keep stereochemistry or strip it. Discuss it with your neighbour (or yourself): there is a defensible answer either way, and the point is the argument.


In [ ]:
s_nap = molecules[0]
r_nap = molecules[1]
print("same with stereo:", Chem.MolToSmiles(s_nap) == Chem.MolToSmiles(r_nap))
print("same without stereo:", Chem.MolToSmiles(_, isomericSmiles=_) == Chem.MolToSmiles(_, isomericSmiles=_))

The decision about keeping stereo chemistry is only worth something once it is applied to the whole table.
The rest of this notebook follows the decision **you argued** in answer 4.

#### Questions

5. Call `clean_catalogue` on the repaired catalogue with stereochemistry information removed. Store the result as `dataset`.
6. Print `dataset` to see what was kept.


In [ ]:
dataset = clean_catalogue(_, keep_stereo=_)
dataset

#### Question

6. Look at the names that survived: `naproxen (S)` and `ibuprofen (racemate)` are still in the table, but the `smiles` column no longer carries any stereochemistry at all. What is misleading about that, and what would you do about it before sending this table to anybody else?


## Part 3 - Do The Numbers Even Matter?

A [molecular descriptor](https://en.wikipedia.org/wiki/Molecular_descriptor) is one number computed from a structure by a fixed rule, and [video 3](https://youtu.be/esyn9kJvkcE)'s central warning was that the rule is lossy, i.e., describing a molecule by that property does not mean we can recognize the molecule if we are only given the molecular weight.

Molecular weight is such an example where we *cannot distinguish* [glucose](https://en.wikipedia.org/wiki/Glucose) from [fructose](https://en.wikipedia.org/wiki/Fructose) and there are many such examples.

### Questions

1. Predict what the block below prints for the two naproxen enantiomers, before running it: will any of the three numbers differ?
2. Run it, then say what the result implies about the decision you took in the end of Part 2.


In [ ]:
for name, mol in zip(["S-naproxen", "R-naproxen"], [s_nap, r_nap]):
    print(name, "MW =", round(Descriptors.MolWt(mol), 2), "LogP =", round(Descriptors.MolLogP(mol), 2), "TPSA =", round(Descriptors.TPSA(mol), 1))

### Construction of the Final Data Table
[Video 4](https://youtu.be/fZOcjqtm5ss) wrapped six descriptor calls in a function called `describe` returning one dictionary per molecule, because a list of dictionaries with identical keys is exactly what `pd.DataFrame` wants.
[The code is available as part of the video lecture material](https://colab.research.google.com/drive/1MeUTYxC8H7K81NjWetcvgM_89RvfWF-J).

For any valid molecule stored as `mol`, the six descriptors are are:

  1) molecular weight in g/mol (`Descriptors.MolWt(mol)`)
  2) the calculated logP (`Descriptors.MolLogP(mol)`)
  3) the topological polar surface area in Å² (`Descriptors.TPSA(mol)`)

and the counts of

  4) hydrogen-bond donors (`Descriptors.NumHDonors(mol)`)
  5) hydrogen-bond acceptors (`Descriptors.NumHAcceptors(mol)`)
  6) number of rotatable bonds. (`Descriptors.NumRotatableBonds(mol)`)

### Question
In this part of the exercise, we will write a new function called `profile` that calculates these interesting properties for us.

3. Write `profile` from scratch: it takes a name and a molecule and returns a dictionary with the keys `name`, `mw`, `logp`, `tpsa`, `hbd`, `hba` and `rotb`. Round `mw` and `tpsa` to one decimal and `logp` to two; the three counts are whole numbers already.


In [ ]:
def profile(name, mol):
    """Return a dictionary of six physicochemical descriptors for one molecule

    :param name: the name of the molecule
    :param mol: the rdkit mol object
    """
    return {"name": name,        # one key per descriptor, name first
            "mw"  : round(_, 1), # molecular weight
            "logp": round(_, 2), # ...
            "tpsa": round(_, 1),
            "hbd" : _,
            "hba" : -,
            "rotb": _}

### Question

4. Build `new_rows` - a list of new rows by calling `profile` once for each structure in the `dataset`.

The `table` as assembled automatically and shown.


In [ ]:
new_rows = []
for name, mol in zip(_, _):
    new_rows.append(_)

table = pd.DataFrame(new_rows)  # a list of rows (dictionaries) becomes a table
table

We can now summarize the statistics using the built-in function `describe` that works on the entire dataset as shown below.

In [ ]:
table.describe().round(2)

#### Questions

5. Compare the `max` row across the six columns. Roughly how many times larger is the largest value in `mw` than the largest value in `hbd`, and why would that matter for any method that measures the distance between two compounds?
6. One compound is responsible for the maximum of both `mw` and `logp`. Which is it, and does its presence in an oral formulary surprise you?

## Part 4 - Triage

[Lipinski's rule of five](https://en.wikipedia.org/wiki/Lipinski%27s_rule_of_five) was derived in 1997 from compounds that had survived into [phase II trials](https://en.wikipedia.org/wiki/Phases_of_clinical_research#Phase_II), and it sets four thresholds:

$$MW \leq 500 \qquad \log P \leq 5 \qquad HBD \leq 5 \qquad HBA \leq 10$$

Two or more violations is taken as a warning that oral absorption will be poor.

#### Questions
1. create four filters in the code below by translating the math above.

The `follows_lipinski` variable is already set up to yield `True` if *all four requirements* are `True` otherwise it will be False.

2. Add a column `lipinski` to `table` that is 1 when all four thresholds hold and 0 otherwise, using the `astype(int)` function on the `follows_lipinski`variable.
3. **NB! Difficult** Add a column `num_violations` to `table` that counts the sum of each individual filter (from 1. just above here) and subtracts that number from 4. *Hint:* you need the `astype(int)` again.


In [ ]:
mw_filter = _
logp_filter = _
hbd_filter = _
hba_filter = _
follows_lipinski = mw_filter & logp_filter & hbd_filter & hba_filter

table["lipinski"] = follows_lipinski.astype(int)
# table["num_violations"] = 4 - mw_filter.astype(int) - _.astype(int) - _.astype(_) - _.astype(_)  # uncomment line to complete
table

A pass-or-fail verdict answers *"is anything badly wrong"*.
[The quantitative estimate of drug-likeness](https://www.nature.com/articles/nchem.1243)(QED), answers a softer question:

> How typical of a marketed oral drug does this look?

By blending eight properties into a single molecular desirability between 0 and 1.

`QED.qed(mol)` takes a molecule, not a row, so the values have to come from `dataset["mol"]`, which lists the molecules in the same order as the rows of `table`.

### Questions

3. Add a `qed` column to `table`, rounded to three decimals, with a list comprehension over the molecules.
4. Print the table and sort the rows so the most drug-like compound is at the top.


In [ ]:
qed_data = []
for molecule in dataset["mol"]:
    qed_data.append(QED.qed(_))

table["qed"] = _
table.sort_values("_", ascending=_).round(3)

#### Questions

5. Aspirin passes all four Lipinski thresholds comfortably, yet it does not top the QED ranking. Explain how a compound can be a clean pass on the rule and still score only moderately on QED.
6. (Amiodarone)[https://en.wikipedia.org/wiki/Amiodarone] fails on two counts and is still a marketed oral drug. If a screening deck of 100 000 virtual compounds landed on your desk tomorrow, would you use the rule of five on it, and would you use it on a single hit that already showed activity in your assay?


That is today's result.
Nine hand-typed strings from a supplier have become seven clean structures, one row per connectivity by a decision you argued for, each with six physicochemical descriptors, a rule-of-five verdict and a QED score.
You can also say where the two verdicts disagree and which one to trust for what: the rule for triaging a large library, neither of them for judging a single compound that already works.

Two things are worth noticing about what the table does *not* contain.
It has no stereochemistry, and it has no measured property of any kind.
Everything in it was computed from connectivity alone, which is what makes it cheap, and also what limits how much it can ever tell you.
